# Pokemon Project Rebuild: Upgraded Notebook + Streamlit Demo

This notebook rebuilds the original IS510 Pokemon project with a cleaner data pipeline, a stronger type-prediction model, a fixed 50k-row battle dataset, and a local Streamlit demo that uses the same training code.

## Step 0. Setup

We reuse a shared Python module so the notebook and Streamlit app always stay aligned.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from pokemon_project import (
    load_local_data,
    summarize_local_data,
    audit_matchup_coverage,
    build_master_table,
    train_project_bundle,
    notebook_ready_tables,
    predict_types,
    predict_battle,
    fetch_pokeapi_enrichment,
    fetch_smogon_usage_stats,
)

ROOT = Path.cwd()
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

## Step 1. Audit The Local Project Assets

This confirms what the original project shipped with and reproduces the key data issue in the old battle workflow.

In [2]:
local_data = load_local_data(ROOT)
display(summarize_local_data(local_data))
display(audit_matchup_coverage(local_data))

,dataset,rows,columns,missing_cells
0,legacy_cn,1216,11,0
1,pokemon,1215,30,7727
2,single_combats,50000,3,0
3,team_combats,10000,3,0
4,team_ids,100,7,0
5,type_matchup,540,20,0


,original_single_combats_rows,rows_after_old_incomplete_matchup_join,rows_lost_by_old_join,coverage_ratio,unique_pokemon_ids_in_combats,unique_ids_covered_by_matchup_csv
0,50000,15819,34181,0.31638,784,441


## Step 2. Optional External Enrichment Hooks

The rebuild supports optional caching from PokeAPI and Smogon. The demo stays fully local at runtime, so these calls are intentionally left commented out by default.

```python
master_preview = build_master_table(ROOT)
fetch_pokeapi_enrichment(master_preview['dexnum_int'].tolist(), cache_path=ROOT / 'pokeapi_enrichment_cache.json')
fetch_smogon_usage_stats(cache_path=ROOT / 'smogon_usage_cache.json')
```

## Step 3. Build The Unified Master Table

This table replaces the older ad-hoc notebook merges and becomes the single source of truth for both tasks.

In [3]:
master_df = build_master_table(ROOT, include_external=False)
display(master_df.head())
print('Master table shape:', master_df.shape)
print('Unique Pokemon by dex number:', master_df['dexnum_int'].nunique())

,dexnum,name,generation,type1,type2,species,height,weight,ability1,ability2,hidden_ability,hp,attack,defense,sp_atk,sp_def,speed,total,ev_yield,catch_rate,base_friendship,base_exp,growth_rate,egg_group1,egg_group2,percent_male,percent_female,egg_cycles,special_group,url,dexnum_int,image_url,bulk_score,offense_score,physical_bias,special_bias,speed_rank_pct,single_type_flag
0,1.0,Bulbasaur,1.0,Grass,Poison,Seed Pokémon,0.7,6.9,Overgrow,Chlorophyll,Unknown,45.0,49.0,49.0,65.0,65.0,45.0,318.0,1 Sp. Atk,45.0,50.0,64.0,Medium Slow,Grass,Monster,87.5,12.5,20.0,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...,1,https://img.pokemondb.net/sprites/scarlet-viol...,159.0,159.0,-16.0,16.0,0.251707,0
1,2.0,Ivysaur,1.0,Grass,Poison,Seed Pokémon,1.0,13.0,Overgrow,Chlorophyll,Unknown,60.0,62.0,63.0,80.0,80.0,60.0,405.0,"1 Sp. Atk, 1 Sp. Def",45.0,50.0,142.0,Medium Slow,Grass,Monster,87.5,12.5,20.0,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...,2,https://img.pokemondb.net/sprites/scarlet-viol...,203.0,202.0,-18.0,18.0,0.433659,0
2,3.0,Venusaur,1.0,Grass,Poison,Seed Pokémon,2.0,100.0,Overgrow,Chlorophyll,Unknown,80.0,82.0,83.0,100.0,100.0,80.0,525.0,"2 Sp. Atk, 1 Sp. Def",45.0,50.0,236.0,Medium Slow,Grass,Monster,87.5,12.5,20.0,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...,3,https://img.pokemondb.net/sprites/scarlet-viol...,263.0,262.0,-18.0,18.0,0.670244,0
3,4.0,Charmander,1.0,Fire,None,Lizard Pokémon,0.6,8.5,Blaze,Solar Power,Unknown,39.0,52.0,43.0,60.0,50.0,65.0,309.0,1 Speed,45.0,50.0,62.0,Medium Slow,Dragon,Monster,87.5,12.5,20.0,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...,4,https://img.pokemondb.net/sprites/scarlet-viol...,132.0,177.0,-8.0,8.0,0.500000,1
4,5.0,Charmeleon,1.0,Fire,None,Flame Pokémon,1.1,19.0,Blaze,Solar Power,Unknown,58.0,64.0,58.0,80.0,65.0,80.0,405.0,"1 Sp. Atk, 1 Speed",45.0,50.0,142.0,Medium Slow,Dragon,Monster,87.5,12.5,20.0,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...,5,https://img.pokemondb.net/sprites/scarlet-viol...,181.0,224.0,-16.0,16.0,0.670244,1


Master table shape: (1025, 38)
Unique Pokemon by dex number: 1025


## Step 4. Train All Candidate Models

This runs baseline reproduction plus the upgraded modeling pipeline for both type prediction and 1v1 battle prediction.

In [4]:
bundle = train_project_bundle(ROOT, include_external=False)
tables = notebook_ready_tables(bundle)
print('Final type model:', bundle['type_bundle']['final_model_name'])
print('Final battle model:', bundle['battle_bundle']['final_model_name'])

Final type model: ClassifierChain Logistic (C=10)
Final battle model: Extra Trees


## Step 5. Type Prediction Results

We report both a random split for continuity with the original course notebook and a stricter grouped split by species tag.

In [5]:
display(tables['type_random'][['model', 'micro_f1', 'macro_f1', 'hamming_loss', 'exact_match']].round(4))
display(tables['type_grouped'][['model', 'micro_f1', 'macro_f1', 'hamming_loss', 'exact_match']].round(4))

,model,micro_f1,macro_f1,hamming_loss,exact_match
0,ClassifierChain Logistic (C=10),0.7326,0.6967,0.0534,0.5073
1,ExtraTrees MultiOutput,0.6824,0.6158,0.0555,0.3122
2,OVR Logistic,0.5260,0.3992,0.0796,0.1268


,model,micro_f1,macro_f1,hamming_loss,exact_match
0,ClassifierChain Logistic (C=10),0.6344,0.5649,0.0709,0.3470
1,ExtraTrees MultiOutput,0.5762,0.4471,0.0675,0.1781
2,OVR Logistic,0.5089,0.3657,0.0798,0.1279


In [6]:
type_examples = []
for name in ['Charizard', 'Gengar', 'Lucario', 'Garchomp']:
    result = predict_types(name, bundle).payload
    type_examples.append({
        'name': name,
        'true_primary': result['true_primary'],
        'true_secondary': result['true_secondary'],
        'pred_primary': result['predicted_primary'],
        'pred_secondary': result['predicted_secondary'],
        'top_prob': float(result['probabilities'].iloc[0]['probability']),
        'model': result['model_name'],
    })
display(pd.DataFrame(type_examples))
predict_types('Charizard', bundle).payload['probabilities'].head(8)

,name,true_primary,true_secondary,pred_primary,pred_secondary,top_prob,model
0,Charizard,Fire,Flying,Fire,Flying,0.990083,ClassifierChain Logistic (C=10)
1,Gengar,Ghost,Poison,Ghost,Poison,0.970199,ClassifierChain Logistic (C=10)
2,Lucario,Fighting,Steel,Fire,Fighting,0.591604,ClassifierChain Logistic (C=10)
3,Garchomp,Dragon,Ground,Dragon,Ground,0.958745,ClassifierChain Logistic (C=10)


,type,probability
0,Fire,0.990083
1,Flying,0.797485
2,Dragon,0.043013
3,Ground,0.017279
4,Poison,0.015114
5,Water,0.002824
6,Fighting,0.002780
7,Electric,0.001254


## Step 6. Battle Prediction Results

The main upgrade here is using the full 50,000 single combats instead of the 15,819 rows that survived the old incomplete type-matchup join.

In [7]:
display(tables['battle_random'][['model', 'feature_set', 'accuracy', 'roc_auc']].round(4))
display(tables['battle_grouped'][['model', 'feature_set', 'accuracy', 'roc_auc']].round(4))

,model,feature_set,accuracy,roc_auc
0,Extra Trees,full,0.8153,0.8958
1,Random Forest,full,0.8088,0.8922
2,Logistic Regression,baseline,0.5358,0.5349


,model,feature_set,accuracy,roc_auc
0,Extra Trees,full,0.8018,0.8756
1,Random Forest,full,0.7946,0.8699
2,Logistic Regression,baseline,0.5377,0.5460


In [8]:
battle_examples = []
for poke_a, poke_b in [('Charizard', 'Blastoise'), ('Pikachu', 'Gyarados'), ('Garchomp', 'Togekiss')]:
    result = predict_battle(poke_a, poke_b, bundle).payload
    battle_examples.append({
        'pokemon_a': poke_a,
        'pokemon_b': poke_b,
        'predicted_winner': result['predicted_winner'],
        'win_prob_a': round(result['win_prob_a'], 4),
        'win_prob_b': round(result['win_prob_b'], 4),
        'historical_battles': result['history']['total_battles'],
        'model': result['model_name'],
    })
display(pd.DataFrame(battle_examples))
predict_battle('Garchomp', 'Togekiss', bundle).payload['feature_snapshot']

,pokemon_a,pokemon_b,predicted_winner,win_prob_a,win_prob_b,historical_battles,model
0,Charizard,Blastoise,Blastoise,0.1800,0.8200,0,Extra Trees
1,Pikachu,Gyarados,Gyarados,0.4314,0.5686,0,Extra Trees
2,Garchomp,Togekiss,Garchomp,0.7543,0.2457,1,Extra Trees


,feature,value
0,A_best_stab,0.0
1,B_best_stab,2.0
2,total_diff,55.0
3,speed_diff,22.0
4,weak_sum_diff,2.0


## Step 7. Streamlit Demo

The local app is implemented in `streamlit_app.py`. Run it from the project folder with:

```bash
streamlit run streamlit_app.py
```

The app exposes two pages:
- `Type Predictor`: sprite, true labels, predicted labels, probability table, profile explanation
- `Battle Predictor`: winner probability, matchup explanation, historical battle lookup, feature snapshot

## Step 8. Final Takeaways

- Type prediction improved from the old notebook baseline into a stronger exact-match regime while still using classical ML.
- Battle prediction now uses the full singles dataset and reaches the target accuracy/AUC range from the implementation plan.
- The notebook and Streamlit app are backed by the same shared module, so there is one consistent training and inference path.